# MSigDB decoupleR ORA Saturation Analysis

**Environment:** `clamp-analyses`

For each CLAMPfull model in the saturation grid (`07_saturation`, varying K across 4 data coverage levels and up to 3 seeds), this notebook:

1. Loads the Z matrix (gene loadings per LV).
2. Filters genes to the model gene universe overlapping MSigDB (v2026.1).
3. Runs `decoupleR::run_ora()` on the full loading matrix at once (all LVs as columns), selecting the top 1% positive-loading genes per LV (`n_up`), no bottom/negative tail (`n_bottom = 0`), with `n_background` set to the filtered model gene universe size.
4. Stores raw `terms_padj`: the minimum BH-adjusted p-value per MSigDB term across all LVs (BH-adjustment done within each LV, then minimum taken across LVs).
5. Saves per-model RDS caches (`_msigdb_decoupler_ora.rds`) for downstream saturation plotting. FDR thresholds and coverage computation are done in `01_msigdb_decoupler_ora_plot.ipynb`.

In [ ]:
library(here)
library(dplyr)
library(decoupleR)

## Paths

In [ ]:
models_dir <- here("output/01_model_building/04_archs4/07_saturation")
output_dir <- here("output/03_model_biology/00_archs4/07_saturation_random/decoupler_ora")

dir.create(file.path(output_dir, "CLAMPfull"), recursive = TRUE, showWarnings = FALSE)

## Model grid

Build the full (rs_pct, k_val, seed) grid and filter to combinations where Z.csv exists.

In [ ]:
rs_pcts  <- c(1L, 5L, 10L, 25L)
k_values <- c(86L, 173L, 432L, 864L, 1296L, 1728L)
seeds    <- 1:3

model_grid <- expand.grid(
  rs_pct = rs_pcts,
  k_val  = k_values,
  seed   = seeds,
  stringsAsFactors = FALSE
)

model_grid$subdir <- sprintf(
  "hall_saturation_rs%d_k%d_seed_%d",
  model_grid$rs_pct, model_grid$k_val, model_grid$seed
)

model_grid$z_path <- file.path(
  models_dir, model_grid$subdir, "CLAMPfull_hall", "Z.csv"
)

model_grid <- model_grid[file.exists(model_grid$z_path), ]
rownames(model_grid) <- NULL

message("Available models: ", nrow(model_grid))
print(model_grid[, c("rs_pct", "k_val", "seed")])

## Load MSigDB gene sets as a decoupleR network

In [ ]:
msig_gmt <- clusterProfiler::read.gmt(here("data/pathways/msigdb.v2026.1.Hs.symbols.gmt"))
net <- msig_gmt %>% dplyr::rename(source = term, target = gene)
message(sprintf("MSigDB gene sets loaded: %d", length(unique(net$source))))

## Helper: run decoupleR ORA for one model

Returns a list with raw `terms_padj` (minimum BH-adjusted p-value per MSigDB term across all LVs).

In [ ]:
run_ora_for_model <- function(z_path) {
  Z              <- read.csv(z_path, row.names = 1, check.names = FALSE)
  universe_genes <- rownames(Z)
  mat            <- as.matrix(Z[universe_genes %in% net$target, , drop = FALSE])
  n_background   <- nrow(mat)
  n_top          <- ceiling(0.01 * n_background)
  n_lvs          <- ncol(mat)

  term_overlap   <- tapply(net$target %in% rownames(mat), net$source, sum)
  n_total_msigdb <- sum(term_overlap >= 10L)

  ora_res <- decoupleR::run_ora(
    mat          = mat,
    network      = net,
    n_up         = n_top,
    n_bottom     = 0,
    n_background = n_background,
    minsize      = 10
  )

  # BH-adjust within each LV (condition), then take min adjusted p per pathway across LVs
  ora_res <- ora_res %>%
    dplyr::group_by(condition) %>%
    dplyr::mutate(p_adj = p.adjust(p_value, method = "BH")) %>%
    dplyr::ungroup()

  # n_samples via B.csv header only (avoids loading the full multi-GB model rds)
  b_path <- file.path(dirname(z_path), "B.csv")
  n_samples <- if (file.exists(b_path)) {
    ncol(read.csv(b_path, nrows = 0, check.names = FALSE))
  } else NA_integer_

  list(
    n_samples      = n_samples,
    n_lvs          = n_lvs,
    n_top_genes    = n_top,
    n_total_msigdb = n_total_msigdb,
    terms_padj     = tapply(ora_res$p_adj, ora_res$source, min)
  )
}

## Helper: build results row from ORA output

In [ ]:
build_row <- function(spec, res) {
  data.frame(
    rs_pct           = spec$rs_pct,
    k_val            = spec$k_val,
    seed             = spec$seed,
    n_samples        = res$n_samples,
    n_lvs            = res$n_lvs,
    n_top_genes      = res$n_top_genes,
    n_total_msigdb   = res$n_total_msigdb,
    stringsAsFactors = FALSE
  )
}

## Run ORA: CLAMPfull

In [ ]:
results_list <- lapply(seq_len(nrow(model_grid)), function(i) {
  spec <- model_grid[i, ]

  cache_path <- file.path(
    output_dir, "CLAMPfull",
    sprintf("rs%d_k%d_seed%d_msigdb_decoupler_ora.rds", spec$rs_pct, spec$k_val, spec$seed)
  )

  if (file.exists(cache_path)) {
    message(sprintf("Loading cached: rs%d k%d seed%d", spec$rs_pct, spec$k_val, spec$seed))
    res <- readRDS(cache_path)
  } else {
    message(sprintf("Running decoupleR ORA: rs%d k%d seed%d", spec$rs_pct, spec$k_val, spec$seed))
    res <- run_ora_for_model(spec$z_path)
    if (!is.null(res)) saveRDS(res, cache_path)
  }

  if (is.null(res)) return(NULL)
  build_row(spec, res)
})

results_df <- do.call(rbind, Filter(Negate(is.null), results_list))
rownames(results_df) <- NULL
results_df <- results_df %>%
  dplyr::arrange(rs_pct, k_val, seed)

message("Collected ", nrow(results_df), " rows")
print(results_df)

In [ ]:
write.csv(
  results_df,
  file.path(output_dir, "results_CLAMPfull_msigdb_decoupler_ora.csv"),
  row.names = FALSE
)
message("Saved results_CLAMPfull_msigdb_decoupler_ora.csv")